# Stochastic Calculus from Zero · Episode 28: Capstone: one AUD bond option, four ways

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **Today's curve** (Part 4): Series 1's illustrative AONIA curve, valued 25 September 2026 (`quotes_illustrative.csv`), with Hull-White mean reversion 3% and volatility 0.94% (Series 4's calibration to its illustrative AUD swaptions, rounded). Shaped like a plausible market, made up for teaching; not market data.
>
> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The trade · 4. Route 1: the closed form · 5. Route 2: risk-neutral paths · 6. Route 3: forward-measure paths · 7. Route 4: the PDE · 8. Four routes, one number · 9. Part 4 checkpoint · 10. Exercises · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026
valuation_date = "2026-09-25"
aud_quotes_file = "quotes_illustrative.csv"

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 28,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative AONIA curve (made up for teaching)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))


# ---- Part 4 additions -------------------------------------------------------------------------------------------
#   Hull-White fitted to today's curve P0(t), in the G1++ form r(t) = x(t) + phi(t), dx = -a x dt + sigma dW, x(0) = 0.
def hw_B(t, T, a):
    return (1 - np.exp(-a * (T - t))) / a


def hw_V(t, T, a, sigma):
    """Variance of the integral of x from t to T, given x(t)."""
    tau = T - t
    return sigma**2 / a**2 * (tau + 2 / a * np.exp(-a * tau) - np.exp(-2 * a * tau) / (2 * a) - 1.5 / a)


def hw_bond(t, T, x, P0, a, sigma):
    """Zero-coupon bond P(t, T) given the state x(t); P0 is today's discount curve."""
    return P0(T) / P0(t) * np.exp(0.5 * (hw_V(t, T, a, sigma) - hw_V(0, T, a, sigma) + hw_V(0, t, a, sigma))
                                  - hw_B(t, T, a) * np.asarray(x))


def hw_phi(t, f0, a, sigma):
    """The deterministic shift: r(t) = x(t) + phi(t) reprices today's curve (f0 = today's instantaneous forwards)."""
    return f0(t) + sigma**2 / (2 * a**2) * (1 - math.exp(-a * t))**2


def hw_exact_step(x, I, dt, a, sigma, z):
    """One exact step of (x, integral of x) over dt; z has shape (n, 2) of independent standard normals."""
    e = math.exp(-a * dt)
    bd = (1 - e) / a
    vx = sigma**2 / (2 * a) * (1 - e * e)
    vi = sigma**2 / a**2 * (dt - 2 * bd + (1 - e * e) / (2 * a))
    cxi = sigma**2 / (2 * a**2) * (1 - e)**2
    eps = z @ np.linalg.cholesky([[vx, cxi], [cxi, vi]]).T
    return x * e + eps[:, 0], I + x * bd + eps[:, 1]


#   Jumps: Merton's model, log jump sizes normal with mean mu_j and standard deviation d_j, lam jumps a year.
def merton_call(s, k, t, r, vol, lam, mu_j, d_j, terms=60):
    """Merton call price: Black-Scholes prices weighted by the Poisson chance of n jumps."""
    kbar = math.exp(mu_j + 0.5 * d_j**2) - 1
    lp = lam * (1 + kbar) * t
    total, w = 0.0, math.exp(-lp)
    for n in range(terms):
        if n:
            w *= lp / n
        total = total + w * bs_call(s, k, t, r - lam * kbar + n * math.log(1 + kbar) / t,
                                    math.sqrt(vol**2 + n * d_j**2 / t))[0]
    return total


def bs_implied_vol(price, s, k, t, r, lo=0.01, hi=1.5):
    """Black-Scholes implied volatility by bisection."""
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float(bs_call(s, k, t, r, mid)[0]) < price else (lo, mid)
    return 0.5 * (lo + hi)

Series 1's AONIA curve (`episodes/_shared/aud_curves.py`). The curve code expects `today`, `cal` and `dc`:

In [ ]:
today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()
out["meta"]["valuation_date"] = valuation_date

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

In [ ]:
curves, handles, _, _ = build_curves(pd.read_csv(aud_quotes_file))
aonia, aonia_h = curves["AONIA"], handles["AONIA"]
P0 = lambda t: aonia.discount(float(t)) if t > 0 else 1.0                                     # today's discount curve
f0 = lambda t: aonia.forwardRate(float(t), float(t), ql.Continuous, ql.NoFrequency).rate()    # instantaneous forwards
A_HW, SIG_HW = 0.03, 0.0094                                                                   # Hull-White, Series 4 (rounded)

## 2. Last episode's answer

$P(N_1 = 0) = e^{-0.5} = 60.7\%$.

In [ ]:
out["prev"] = {"answer_pct": math.exp(-0.5) * 100}

## 3. The trade

A European call, expiring in 1 year, on the zero-coupon bond paying 1 in 5 years, discounted on Series 1's AONIA curve. The strike is the forward bond price $P(0, 5)/P(0, 1)$. The model is Episode 26's Hull-White, fitted to today's curve: $r = x + \varphi(t)$, $dx = -ax\,dt + \sigma\,dW$, $a = 3\%$, $\sigma = 0.94\%$. At expiry the bond is worth `hw_bond(1, 5, x)`, so each route only has to get the distribution of $x(1)$ and the discounting right.

In [ ]:
T1, T2 = 1.0, 5.0
P01, P05 = P0(T1), P0(T2)
F0 = P05 / P01
Kb = F0
trade = {"T1": T1, "T2": T2, "P01": P01, "P05": P05, "forward": F0, "strike": Kb, "a_pct": A_HW * 100, "sigma_pct": SIG_HW * 100}
out["trade"] = trade
trade

## 4. Route 1: the closed form

Under the 1-year forward measure (Episode 25) the forward bond price $F_t = P(t, 5)/P(t, 1)$ is a martingale, and in Hull-White it is lognormal with total volatility

$$v = \frac{\sigma}{a}\big(1 - e^{-a(T_2 - T_1)}\big)\sqrt{\frac{1 - e^{-2aT_1}}{2a}} .$$

So the call is Black's formula on the forward, discounted once: $P(0, T_1)\big(F_0N(d_1) - KN(d_2)\big)$ (Jamshidian's formula). QuantLib's `HullWhite.discountBondOption` should agree.

In [ ]:
v = SIG_HW / A_HW * (1 - math.exp(-A_HW * (T2 - T1))) * math.sqrt((1 - math.exp(-2 * A_HW * T1)) / (2 * A_HW))
d1 = (math.log(F0 / Kb) + 0.5 * v * v) / v
black = P01 * (F0 * float(norm_cdf(d1)) - Kb * float(norm_cdf(d1 - v)))
hw = ql.HullWhite(aonia_h, A_HW, SIG_HW)
ql_call = hw.discountBondOption(ql.Option.Call, Kb, T1, T2)
assert abs(black - ql_call) < 1e-10
out["route1"] = {"vol_pct": v * 100, "black": black, "ql": ql_call, "bp": black * 1e4}
out["route1"]

## 5. Route 2: risk-neutral paths

Under the risk-neutral measure (Episode 21) the price is $E\big[e^{-\int_0^1 r\,dt}\max(P(1, 5) - K, 0)\big]$. The pair $(x(1), \int_0^1 x\,dt)$ is jointly normal, so one exact step simulates both (`hw_exact_step`), and $e^{-\int_0^1 r} = P(0, 1)\,e^{-\frac12V(0, 1) - \int_0^1 x}$.

In [ ]:
rng = np.random.default_rng(seed)
n = 200_000
x1, I1 = hw_exact_step(np.zeros(n), np.zeros(n), T1, A_HW, SIG_HW, rng.standard_normal((n, 2)))
disc = P01 * np.exp(-0.5 * hw_V(0, T1, A_HW, SIG_HW) - I1)
pay_rn = disc * np.maximum(hw_bond(T1, T2, x1, P0, A_HW, SIG_HW) - Kb, 0)
out["route2"] = {"paths": n, "price": float(pay_rn.mean()), "se": float(pay_rn.std() / math.sqrt(n)),
                 "bp": float(pay_rn.mean() * 1e4), "se_bp": float(pay_rn.std() / math.sqrt(n) * 1e4),
                 "disc_check": float(disc.mean()), "P01": P01}
assert abs(out["route2"]["price"] - ql_call) < 3 * out["route2"]["se"]
assert abs(disc.mean() / P01 - 1) < 5e-4            # the discount factors average to today's 1-year bond
out["route2"]

## 6. Route 3: forward-measure paths

Under the 1-year forward measure (Episode 25) the price is $P(0, 1)\,E^1\big[\max(P(1, 5) - K, 0)\big]$: discount once. Girsanov shifts the mean of $x(1)$ by $-\sigma^2\int_0^1 e^{-a(1 - s)}B(s, 1)\,ds$, which works out to $-\frac{\sigma^2}{a^2}(1 - e^{-a}) + \frac{\sigma^2}{2a^2}(1 - e^{-2a})$; its variance is unchanged.

In [ ]:
mean_f = -(SIG_HW**2 / A_HW**2 * (1 - math.exp(-A_HW * T1)) - SIG_HW**2 / (2 * A_HW**2) * (1 - math.exp(-2 * A_HW * T1)))
sd = SIG_HW * math.sqrt((1 - math.exp(-2 * A_HW * T1)) / (2 * A_HW))
xf = mean_f + sd * rng.standard_normal(n)
bond_f = hw_bond(T1, T2, xf, P0, A_HW, SIG_HW)
pay_f = P01 * np.maximum(bond_f - Kb, 0)
out["route3"] = {"paths": n, "price": float(pay_f.mean()), "se": float(pay_f.std() / math.sqrt(n)),
                 "bp": float(pay_f.mean() * 1e4), "se_bp": float(pay_f.std() / math.sqrt(n) * 1e4),
                 "mean_shift_bp": mean_f * 1e4, "E_bond": float(bond_f.mean()), "E_bond_se": float(bond_f.std() / math.sqrt(n))}
assert abs(out["route3"]["price"] - ql_call) < 3 * out["route3"]["se"]
assert abs(bond_f.mean() - F0) < 3 * out["route3"]["E_bond_se"] + 1e-6      # the forward is a martingale
out["route3"]

## 7. Route 4: the PDE

Feynman-Kac (Episode 23) in the state $x$: $u(t, x)$ = the option's value solves

$$u_t - a\,x\,u_x + \tfrac12\sigma^2u_{xx} - \big(x + \varphi(t)\big)u = 0, \qquad u(1, x) = \max\big(P(1, 5; x) - K, 0\big),$$

with $\varphi(t) = f(0, t) + \frac{\sigma^2}{2a^2}(1 - e^{-at})^2$ (`hw_phi`). Crank-Nicolson on $x \in [-15\%, 15\%]$, 401 points, 400 time steps, read off at $x = 0$.

In [ ]:
xg = np.linspace(-0.15, 0.15, 401); hx = xg[1] - xg[0]; M = 400; k = T1 / M
u = np.maximum(hw_bond(T1, T2, xg, P0, A_HW, SIG_HW) - Kb, 0)
I = np.eye(xg.size)
diff = 0.5 * SIG_HW**2
for m in range(M, 0, -1):
    tm = (m - 0.5) * k
    drift, disc_rate = -A_HW * xg, xg + hw_phi(tm, f0, A_HW, SIG_HW)
    L = (np.diag(-2 * diff / hx**2 - disc_rate) + np.diag((diff / hx**2 - drift / (2 * hx))[1:], -1)
         + np.diag((diff / hx**2 + drift / (2 * hx))[:-1], 1))
    L[0, :] = 0; L[-1, :] = 0; L[0, 0] = -disc_rate[0]; L[-1, -1] = -disc_rate[-1]
    u = np.linalg.solve(I - 0.5 * k * L, (I + 0.5 * k * L) @ u)
pde = float(np.interp(0.0, xg, u))
assert abs(pde - ql_call) < 2e-6
out["route4"] = {"price": pde, "bp": pde * 1e4, "points": xg.size, "steps": M}
out["route4"]

## 8. Four routes, one number

| Route | Tools | From |
|---|---|---|
| Closed form | Forward measure, lognormal martingale, Black's formula | Episodes 21, 25 |
| Risk-neutral paths | Itô integral of $x$, exact OU step, discounting inside the average | Episodes 13, 18, 21 |
| Forward-measure paths | Change of numéraire, Girsanov's mean shift | Episodes 21, 25 |
| PDE | Feynman-Kac, Crank-Nicolson | Episode 23 |

All rest on Part 1's martingales and Part 2's Itô calculus.

In [ ]:
out["all"] = {"rows": [
    {"route": "Closed form (Black on the forward)", "bp": out["route1"]["bp"], "se_bp": None},
    {"route": "QuantLib HullWhite.discountBondOption", "bp": ql_call * 1e4, "se_bp": None},
    {"route": "Risk-neutral paths", "bp": out["route2"]["bp"], "se_bp": out["route2"]["se_bp"]},
    {"route": "Forward-measure paths", "bp": out["route3"]["bp"], "se_bp": out["route3"]["se_bp"]},
    {"route": "PDE (Crank-Nicolson)", "bp": out["route4"]["bp"], "se_bp": None}],
    "spread_bp": max(abs(r - ql_call * 1e4) for r in (out["route2"]["bp"], out["route3"]["bp"], out["route4"]["bp"]))}
pd.DataFrame(out["all"]["rows"])

## 9. Part 4 checkpoint

Try these; the answers are in the cell below (and in the book).

1. For $\sigma(t, T) = 0.01$ (constant), write the HJM drift.
2. A Poisson process has $\lambda = 2$ a year. What are the mean and variance of $N_3$, and $P(N_1 = 0)$?
3. Why can't a delta hedge remove jump risk? What extra instrument would help?
4. Name the four routes to this episode's price, and the episode each relies on.

In [ ]:
out["checkpoint"] = {"q1": "alpha = 0.0001 (T - t)", "q2_mean": 6, "q2_var": 6, "q2_p0_pct": math.exp(-2) * 100}
print("1. alpha(t, T) = 0.01 * 0.01 (T - t) = 0.0001 (T - t): Ho-Lee (Episode 26)")
print(f"2. mean = variance = 2 x 3 = 6; P(N_1 = 0) = e^-2 = {math.exp(-2):.2%} (Episode 27)")
print("3. A jump moves the share by a finite amount between rebalances, and the option's curvature turns it into a loss;"
      " a second option, which also responds to jumps, can hedge it (Episode 27)")
print("4. Closed form (Episode 25), risk-neutral paths (Episode 21), forward-measure paths (Episode 25), PDE (Episode 23)")

## 10. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Derive the volatility of $F_t = P(t, 5)/P(t, 1)$ in Hull-White: show $d\ln F$ has volatility $\sigma\big(B(t, 5) - B(t, 1)\big)$, and integrate its square from 0 to 1.

<details><summary>Answer</summary>

Each bond has volatility $-\sigma B(t, T)$, so the ratio has $-\sigma(B(t, 5) - B(t, 1)) = -\frac{\sigma}{a}e^{-a(1 - t)}(1 - e^{-4a})$. Squaring and integrating over $[0, 1]$: $\frac{\sigma^2}{a^2}(1 - e^{-4a})^2\frac{1 - e^{-2a}}{2a}$, the square of the $v$ in Section 4.

</details>

**Exercise 2 (By hand).** Show put-call parity for the bond option: call − put $= P(0, 5) - K\,P(0, 1)$. With the strike at the forward, what does that make the put?

<details><summary>Answer</summary>

Call − put pays $P(1, 5) - K$ at time 1, worth $P(0, 5) - KP(0, 1)$. With $K = P(0, 5)/P(0, 1)$ this is 0: the put equals the call.

</details>

**Exercise 3 (By hand).** For a *Bermudan* option (exercisable on several dates), which of the four routes still works directly, and why?

<details><summary>Answer</summary>

The PDE (or a tree): stepping backwards, at each exercise date replace $u$ by $\max(u, \text{exercise value})$. The closed form has no early exercise, and plain forward Monte Carlo doesn't know the continuation value (it needs regression, as in Series 4's AMC).

</details>

**Exercise 4 (In code).** Price the matching put four ways: QuantLib, Black on the forward, forward-measure paths and parity.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
ql_put = hw.discountBondOption(ql.Option.Put, Kb, T1, T2)
black_put = P01 * (Kb * float(norm_cdf(-(d1 - v))) - F0 * float(norm_cdf(-d1)))
mc_put = P01 * np.maximum(Kb - bond_f, 0)
parity = ql_call - (P05 - Kb * P01)
se = mc_put.std() / math.sqrt(n)
assert abs(ql_put - black_put) < 1e-10 and abs(ql_put - parity) < 1e-10 and abs(mc_put.mean() - ql_put) < 3 * se
print(f"put: QuantLib {ql_put * 1e4:.2f} bp, Black {black_put * 1e4:.2f}, parity {parity * 1e4:.2f}, "
      f"forward-measure paths {mc_put.mean() * 1e4:.2f} ± {se * 1e4:.2f}")

**Exercise 5 (In code).** A caplet on the 3-month simple forward rate from 1 year to 1.25 years, strike 4%, pays $\tau\max(L - K, 0)$ at 1.25 years. Show it equals $(1 + K\tau)$ puts on $P(1, 1.25)$ with strike $1/(1 + K\tau)$, by risk-neutral paths.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
tau, Kc = 0.25, 0.04
P_1_125 = hw_bond(T1, T1 + tau, x1, P0, A_HW, SIG_HW)
L = (1 / P_1_125 - 1) / tau                               # the simple forward rate fixed at 1 year
caplet_mc = disc * P_1_125 * tau * np.maximum(L - Kc, 0)   # paid at 1.25: discount to 1 by P(1, 1.25)
caplet_bond = (1 + Kc * tau) * hw.discountBondOption(ql.Option.Put, 1 / (1 + Kc * tau), T1, T1 + tau)
se = caplet_mc.std() / math.sqrt(n)
assert abs(caplet_mc.mean() - caplet_bond) < 3 * se
print(f"caplet: paths {caplet_mc.mean() * 1e4:.3f} ± {se * 1e4:.3f} bp; (1 + K tau) x bond put {caplet_bond * 1e4:.3f} bp")

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")